# 01 · Data audit

Before estimating anything: does the data match its published description, and where are
the traps? This notebook writes `reports/metrics/01_audit.json`, two figures, and the
per-search flags (`data/interim/search_flags.parquet`) that define the analysis sample for
every later notebook.

**The search-level work is SQL.** One row per search, the arm summary, the reserved-slot check,
the data-quality flags and the analysis sample come from the queries in [`sql/`](../sql/), run
by DuckDB directly on the Parquet file, as they would be in a warehouse. Row-level statistics
(prices, balance checks) and the figures stay in pandas. `tests/test_sql.py` checks every SQL
rule against an independent pandas implementation in `hotelrank.audit`.

Without the real data (CI, or anyone who has not downloaded it) the notebook runs on the
synthetic table from `hotelrank.synthetic` and writes nothing to `reports/`.

In [1]:
import sys
from pathlib import Path

def _project_root() -> Path:
    """The folder holding src/hotelrank, wherever the script or notebook was started."""
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / "src" / "hotelrank" / "__init__.py").exists():
            return p
    for marker in here.glob("*/src/hotelrank/__init__.py"):
        return marker.parents[2]
    raise FileNotFoundError("run from inside the project folder")


sys.path.insert(0, str(_project_root() / "src"))

import matplotlib.pyplot as plt
import pandas as pd

from hotelrank import audit, sql
from hotelrank.data import FIGURES_DIR, PARQUET, PRICE_CAP, PRICE_SORT_RHO, RESERVED_SLOTS, ROOT as PROJECT_ROOT, load
from hotelrank.records import show_verdict, write_metrics

COLUMNS = ["srch_id", "date_time", "site_id", "prop_id", "srch_destination_id", "position", "random_bool",
           "click_bool", "booking_bool", "price_usd", "srch_length_of_stay", "prop_starrating",
           "prop_review_score", "prop_location_score1", "prop_location_score2", "promotion_flag",
           "visitor_hist_starrating", "visitor_hist_adr_usd", "srch_query_affinity_score",
           "orig_destination_distance", "gross_bookings_usd"]
df, SOURCE = load(COLUMNS)
con = sql.connect(PARQUET if SOURCE == "real" else df)  # builds the views in sql/00_views.sql
print(f"source: {SOURCE}, {len(df):,} impressions")

source: real, 9,917,530 impressions


## 1. Shape and the two arms

`random_bool = 1` marks searches whose results Expedia showed in random order.

In [2]:
print(sql.SQL_DIR.joinpath("01_arm_summary.sql").read_text(encoding="utf-8"))
searches = sql.search_flags(con)  # one row per search, with the data-quality flags used in section 4
shape = {
    "impressions": int(len(df)),
    "searches": int(len(searches)),
    "hotels": int(df["prop_id"].nunique()),
    "destinations": int(df["srch_destination_id"].nunique()),
    "sites": int(df["site_id"].nunique()),
    "date_min": str(df["date_time"].min().date()),
    "date_max": str(df["date_time"].max().date()),
}
arms = sql.arm_summary(con)
print(shape)
pd.DataFrame(arms)

-- Notebook 01, section 1: searches, impressions and engagement in each arm.
SELECT arm,
       COUNT(*)                                                   AS searches,
       SUM(n_results)                                             AS impressions,
       ROUND(COUNT(*) / SUM(COUNT(*)) OVER (), 4)                 AS share_of_searches,
       ROUND(AVG(CASE WHEN clicks > 0 THEN 1.0 ELSE 0.0 END), 4)  AS searches_with_click,
       ROUND(AVG(CASE WHEN bookings > 0 THEN 1.0 ELSE 0.0 END), 4) AS searches_with_booking,
       ROUND(AVG(clicks), 4)                                      AS clicks_per_search,
       MEDIAN(n_results)                                          AS median_results
FROM searches
GROUP BY arm
ORDER BY arm;



{'impressions': 9917530, 'searches': 399344, 'hotels': 136886, 'destinations': 23715, 'sites': 34, 'date_min': '2012-11-01', 'date_max': '2013-06-30'}


,algorithm,random
searches,2.777980e+05,1.215460e+05
impressions,6.977878e+06,2.939652e+06
share_of_searches,6.956000e-01,3.044000e-01
searches_with_click,1.000000e+00,1.000000e+00
searches_with_booking,9.386000e-01,1.304000e-01
clicks_per_search,1.104100e+00,1.126800e+00
median_results,3.000000e+01,2.800000e+01


Every search has at least one click: the release contains only searches with a click, so
every result in this project describes engaged searches. Each search has at most one booking.

The share of searches ending in a booking differs enormously between the arms (about 94% against
13%). A gap that size is very unlikely to come from the ordering alone and most likely reflects
how the release was sampled; the documentation does not say. Either way the arms are not
comparable on booking rates, so **booking rates are never compared between arms**; every
comparison is made within an arm.

In [3]:
selection = {
    "searches_with_click": round(float((searches["clicks"] > 0).mean()), 4),
    "max_bookings_per_search": int(searches["bookings"].max()),
    "bookings_without_click": int((df["booking_bool"] & ~df["click_bool"]).sum()),
}
print(selection)

{'searches_with_click': 1.0, 'max_bookings_per_search': 1, 'bookings_without_click': 0}


## 2. Positions: four slots are (almost) never filled

Slots 5, 11, 17 and 23 hold a few percent of the impressions of their neighbours in both
arms, and most searches skip exactly four slots. Those slots were presumably used for
something not released (an advert or a different unit). Positions are kept as logged; the
position-effect notebook works with displayed slots and reports the reserved ones separately.

In [4]:
print(sql.SQL_DIR.joinpath("02_reserved_slots.sql").read_text(encoding="utf-8"))
reserved = sql.reserved_slots(con)
occ = audit.slot_occupancy(df)  # every slot, for the figure
gaps = audit.gaps_per_search(searches)
print("reserved slots, impressions relative to neighbours:", reserved)
print("skipped slots per search:", dict(list(gaps.items())[:6]))

fig, ax = plt.subplots(figsize=(9, 3.2))
for arm, colour in (("algorithm", "#1f77b4"), ("random", "#ff7f0e")):
    ax.plot(occ.index, occ[arm] / occ[arm].iloc[0], marker="o", ms=3, lw=1.2, label=f"{arm} order", color=colour)
for s in RESERVED_SLOTS:
    ax.axvline(s, color="grey", lw=0.6, ls=":")
ax.set(xlabel="position", ylabel="impressions, relative to position 1",
       title="Slots 5, 11, 17 and 23 are almost never filled")
ax.legend(frameon=False)
fig.tight_layout()

-- Notebook 01, section 2: how full each reserved slot is relative to its two neighbours.
-- A slot with no impressions still gets a row (count 0), as in audit.slot_occupancy.
WITH counts AS (
    SELECT CASE WHEN CAST(random_bool AS BOOLEAN) THEN 'random' ELSE 'algorithm' END AS arm, position, COUNT(*) AS n
    FROM impressions
    GROUP BY ALL
),
grid AS (
    SELECT a.arm, g.position
    FROM (SELECT DISTINCT arm FROM counts) a,
         generate_series(1, (SELECT MAX(position) FROM impressions)) AS g(position)
),
filled AS (
    SELECT g.arm, g.position, COALESCE(c.n, 0) AS n
    FROM grid g LEFT JOIN counts c USING (arm, position)
),
ratios AS (
    SELECT arm, position, n,
           ROUND(n / ((LAG(n) OVER w + LEAD(n) OVER w) / 2.0), 4) AS vs_neighbours
    FROM filled
    WINDOW w AS (PARTITION BY arm ORDER BY position)
)
SELECT arm, position, n, vs_neighbours
FROM ratios
WHERE position IN (SELECT position FROM reserved_slots)
ORDER BY arm, position;



reserved slots, impressions relative to neighbours: {'algorithm': {5: 0.0614, 11: 0.0835, 17: 0.0939, 23: 0.1001}, 'random': {5: 0.016, 11: 0.0184, 17: 0.0202, 23: 0.0208}}
skipped slots per search: {0: 4564, 1: 40458, 2: 35043, 3: 35287, 4: 278421, 5: 1887}


## 3. Prices

Prices are mostly per night (the median rises only modestly with length of stay), with a
long tail of errors. Rule: a search containing any price outside (0, $5,000] is dropped
from the analysis sample. Later notebooks use price relative to the other hotels in the
same search, which removes currency and per-night/per-stay differences.

In [5]:
prices = audit.price_summary(df)
print(prices)

{'quantiles': {'0.01': 34.0, '0.05': 51.0, '0.25': 85.0, '0.5': 122.07, '0.75': 185.0, '0.95': 356.0, '0.99': 599.0, '0.999': 2132.86}, 'max': 19726328.0, 'impressions_price_le_0': 54, 'share_above_cap': 0.000575, 'cap': 5000.0, 'share_above_10x_search_median': 0.000326, 'median_by_length_of_stay': {1: 112.0, 2: 125.0, 3: 132.8, 4: 141.0, 5: 145.0, 6: 148.0, 7: 144.6}}


## 4. Is the random arm actually random?

Under true random ordering, a hotel's position says nothing about its attributes. The check
demeans each attribute within its search (so differences between markets cancel) and
correlates it with position. The algorithm arm should show strong correlations; the random
arm should show none.

Two problems turn up in the real data:

1. **Some searches are sorted by price.** Their prices rise with position almost perfectly
   (about 5% of random-arm and 4% of algorithm-arm searches), most likely because the user
   chose "sort by price". They are flagged and dropped from both arms.
2. **A residual tilt remains in the random arm**: hotels with more stars sit slightly higher.
   It is diffuse rather than a few re-sorted searches (removing searches sorted by star rating
   barely changes it) and is about 40% of the algorithm arm's tilt, but it is not zero. The
   position-effect notebook therefore adjusts for hotel attributes and reports the estimate
   with and without that adjustment.

In [6]:
flags = searches  # bad_price, price_rho and price_sorted come from sql/00_views.sql
BALANCE_COLS = ["prop_starrating", "prop_review_score", "prop_location_score1", "promotion_flag", "price_usd"]
keep_price_ok = ~flags["bad_price"]
keep_clean = keep_price_ok & ~flags["price_sorted"].fillna(False)

price_sorted_share = {
    name: round(float(flags.loc[(flags["random"] == flag) & flags["price_rho"].notna(), "price_sorted"].mean()), 4)
    for flag, name in audit.ARM_NAMES.items()
}
balance_all = audit.balance(df, BALANCE_COLS, keep=keep_price_ok)
balance_clean = audit.balance(df, BALANCE_COLS, keep=keep_clean)
# Sensitivity: also drop searches whose star ratings fall almost perfectly with position.
flags["star_rho"] = audit.within_search_rank_corr(df, "prop_starrating").reindex(flags.index)
star_sorted = flags["star_rho"] <= -0.9
balance_no_star_sorted = audit.balance(df, BALANCE_COLS, keep=keep_clean & ~star_sorted.fillna(False))
star_sorted_share = {name: round(float(star_sorted[(flags["random"] == flag) & flags["star_rho"].notna()].mean()), 4)
                     for flag, name in audit.ARM_NAMES.items()}
print("share of searches sorted by price:", price_sorted_share)
print("share sorted by stars (rho <= -0.9):", star_sorted_share,
      "random-arm star tilt without them:", balance_no_star_sorted["random"]["prop_starrating"])
pd.DataFrame({"algorithm": balance_all["algorithm"], "random (all)": balance_all["random"],
              "random (price-sorted removed)": balance_clean["random"]})

share of searches sorted by price: {'algorithm': 0.0364, 'random': 0.0549}
share sorted by stars (rho <= -0.9): {'algorithm': 0.0062, 'random': 0.0029} random-arm star tilt without them: -0.097


,algorithm,random (all),random (price-sorted removed)
prop_starrating,-0.2406,-0.0765,-0.0980
prop_review_score,-0.1463,-0.0240,-0.0366
prop_location_score1,-0.1103,-0.0113,-0.0158
promotion_flag,-0.2148,-0.0296,-0.0279
price_usd,-0.0033,-0.0004,-0.0199


In [7]:
fig2, ax2 = plt.subplots(figsize=(7, 3.4))
series = {
    "algorithm order": audit.balance_by_bucket(df, "prop_starrating", False, keep=keep_clean),
    "random order": audit.balance_by_bucket(df, "prop_starrating", True, keep=keep_clean),
}
labels = [f"{int(i.left) + 1}–{int(i.right)}" for i in series["algorithm order"].index]
width = 0.38
for i, (name, s) in enumerate(series.items()):
    ax2.bar([j + (i - 0.5) * width for j in range(len(s))], s.values, width, label=name,
            color=["#1f77b4", "#ff7f0e"][i])
ax2.axhline(0, color="black", lw=0.8)
ax2.set_xticks(range(len(labels)), labels)
ax2.set(xlabel="position", ylabel="stars minus search average",
        title="Random order removes most, not all, of the star-rating tilt")
ax2.legend(frameon=False)
fig2.tight_layout()

## 5. Analysis sample and missing data

In [8]:
print(sql.SQL_DIR.joinpath("03_analysis_sample.sql").read_text(encoding="utf-8"))
sample = {
    "rule": f"drop searches with any price outside (0, {PRICE_CAP:,.0f}]; drop searches sorted by price "
            f"(within-search rank correlation of price with position >= {PRICE_SORT_RHO})",
    **sql.analysis_sample(con),
}
missing = audit.missing_share(df)
print(sample)
print(missing)

-- Notebook 01, section 5: which searches enter the analysis, and why the others do not.
SELECT
    SUM(CASE WHEN bad_price THEN 1 ELSE 0 END)                         AS searches_bad_price,
    SUM(CASE WHEN price_sorted AND NOT bad_price THEN 1 ELSE 0 END)    AS searches_price_sorted,
    SUM(CASE WHEN keep AND arm = 'algorithm' THEN 1 ELSE 0 END)        AS kept_algorithm,
    SUM(CASE WHEN keep AND arm = 'random' THEN 1 ELSE 0 END)           AS kept_random,
    (SELECT COUNT(*) FROM impressions JOIN search_flags USING (srch_id) WHERE keep) AS impressions_kept
FROM search_flags;



{'rule': 'drop searches with any price outside (0, 5,000]; drop searches sorted by price (within-search rank correlation of price with position >= 0.95)', 'searches_bad_price': 538, 'searches_price_sorted': 16773, 'searches_kept': {'algorithm': 267352, 'random': 114681}, 'impressions_kept': 9497418}
{'gross_bookings_usd': 0.9721, 'visitor_hist_starrating': 0.9491, 'visitor_hist_adr_usd': 0.9488, 'srch_query_affinity_score': 0.9359, 'orig_destination_distance': 0.3243, 'prop_location_score2': 0.2196, 'prop_review_score': 0.0015}


In [9]:
record = {
    "shape": shape, "arms": arms, "selection": selection,
    "reserved_slots_vs_neighbours": reserved, "skipped_slots_per_search": gaps,
    "prices": prices, "price_sorted_share": price_sorted_share,
    "balance_within_search_corr": {"price_ok": balance_all, "clean": balance_clean,
                                   "clean_without_star_sorted": balance_no_star_sorted},
    "star_sorted_share": star_sorted_share,
    "stars_minus_search_mean_by_position": {k: {str(i): round(float(v), 4) for i, v in s.items()}
                                            for k, s in series.items()},
    "analysis_sample": sample, "missing_share": missing,
}
path = write_metrics("01_audit", record, SOURCE)
print("wrote", path.relative_to(PROJECT_ROOT) if SOURCE == "real" else path.name)
if SOURCE == "real":
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURES_DIR / "01_reserved_slots.png", dpi=150)
    fig2.savefig(FIGURES_DIR / "01_random_arm_balance.png", dpi=150)
    flags.reset_index().rename(columns={"index": "srch_id"}).to_parquet(
        PROJECT_ROOT / "data" / "interim" / "search_flags.parquet", index=False)

wrote reports/metrics/01_audit.json


## Verdict

What the audit establishes, and the rules the later phases inherit. The verdict is rendered from `record`, the same dictionary written to
`reports/metrics/01_audit.json`, so the summary cannot drift from the analysis above.

In [10]:
a, r = record["arms"]["algorithm"], record["arms"]["random"]
corr = record["balance_within_search_corr"]["clean"]
s = record["analysis_sample"]
fill = [v for arm in record["reserved_slots_vs_neighbours"].values() for v in arm.values()]
verdict = [
    f"{'The data matches the published release' if SOURCE == 'real' else 'Synthetic stand-in data'}: {shape['impressions']:,} impressions in {shape['searches']:,} searches, "
    f"{shape['date_min']} to {shape['date_max']}. Every search has a click and at most one booking, so this is a "
    "selected sample of engaged searches: all rates are conditional on at least one click.",
    f"The two arms are not comparable on booking: {a['searches_with_booking']:.1%} of algorithm-order searches book "
    f"against {r['searches_with_booking']:.1%} of random-order searches. Every comparison is made within an arm.",
    f"Positions {', '.join(map(str, RESERVED_SLOTS))} are filled at only {min(fill):.1%}-{max(fill):.1%} of their "
    "neighbours' rate. They are reserved slots, so they are left out of position curves and choice sets.",
    f"{record['price_sorted_share']['algorithm']:.1%} of algorithm-order and {record['price_sorted_share']['random']:.1%} "
    "of random-order searches are sorted by price: the user re-sorted, so position no longer reflects the ranking "
    "being studied. They are dropped.",
    f"The random arm is close to random, not perfectly random: within a search, corr(position, stars) is "
    f"{corr['random']['prop_starrating']:+.3f} against {corr['algorithm']['prop_starrating']:+.3f} in algorithm order. "
    "Phase 2 therefore reports attribute-adjusted curves next to the raw ones.",
    f"Analysis sample: {s['searches_kept']['algorithm']:,} algorithm-order and {s['searches_kept']['random']:,} "
    f"random-order searches ({s['impressions_kept']:,} impressions), after dropping {s['searches_bad_price']:,} searches "
    f"with impossible prices and {s['searches_price_sorted']:,} price-sorted searches.",
]
show_verdict("Phase 1 verdict: data audit", verdict,
             "Phase 2 uses the random arm to measure how much position itself changes clicks and bookings.")

### Phase 1 verdict: data audit

1. The data matches the published release: 9,917,530 impressions in 399,344 searches, 2012-11-01 to 2013-06-30. Every search has a click and at most one booking, so this is a selected sample of engaged searches: all rates are conditional on at least one click.
2. The two arms are not comparable on booking: 93.9% of algorithm-order searches book against 13.0% of random-order searches. Every comparison is made within an arm.
3. Positions 5, 11, 17, 23 are filled at only 1.6%-10.0% of their neighbours' rate. They are reserved slots, so they are left out of position curves and choice sets.
4. 3.6% of algorithm-order and 5.5% of random-order searches are sorted by price: the user re-sorted, so position no longer reflects the ranking being studied. They are dropped.
5. The random arm is close to random, not perfectly random: within a search, corr(position, stars) is -0.098 against -0.260 in algorithm order. Phase 2 therefore reports attribute-adjusted curves next to the raw ones.
6. Analysis sample: 267,352 algorithm-order and 114,681 random-order searches (9,497,418 impressions), after dropping 538 searches with impossible prices and 16,773 price-sorted searches.

**Next:** Phase 2 uses the random arm to measure how much position itself changes clicks and bookings.